In [ ]:
!pip uninstall -y torchaudio
!pip install -q vllm guidellm
!nvidia-smi

Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
Wed Sep  9 03:01:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P0             26W /   70W |    3145MiB /  15360MiB |      0%      Default |
|               

In [ ]:
!vllm bench throughput --model Qwen/Qwen3-4B-AWQ \
  --dataset-name random --random-input-len 128 --random-output-len 64 --num-prompts 64

When dataset path is not set, it will default to random dataset
/usr/local/lib/python3.13/dist-packages/vllm/benchmarks/throughput.py:1016: UserWarning: Both --prefix-len and --random-prefix-len are specified. The random version (--random-prefix-len) will be preferred in this run.
  validate_args(args)
INFO 09-09 03:04:51 [utils.py:90] Sampling input_len from [128, 128] and output_len from [64, 64]
INFO 09-09 03:04:51 [api_utils.py:272] non-default args: {'tokenizer': 'Qwen/Qwen3-4B-AWQ', 'mm_device_do_normalize': None, 'enable_lora': None, 'reasoning_parser_plugin': '', 'enable_bf16x3_router_gemm': False, 'model': 'Qwen/Qwen3-4B-AWQ'}
INFO 09-09 03:04:52 [model.py:672] Resolved architecture: Qwen3ForCausalLM
INFO 09-09 03:04:52 [model.py:1965] Using max model len 40960
INFO 09-09 03:04:53 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-09 03:04:54 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=[

In [ ]:
# Cell 2 — serve
!nohup vllm serve Qwen/Qwen3-4B-AWQ --dtype float16 > vllm.log 2>&1 &
!curl --retry 60 --retry-delay 10 --retry-all-errors -s http://localhost:8000/health

In [8]:
!guidellm run --backend kind=openai_http,target=http://localhost:8000 \
  --profile kind=sweep --constraint kind=max_duration,seconds=60 \
  --data kind=synthetic_text,prompt_tokens=128,output_tokens=64

Streaming output truncated to the last 5000 lines.
│ [0… synch… (… Req:    0.9 req/s,    1.09s Lat,     1.0 Conc,      55 Comp, … │
│               Tok:   59.6 gen/s,  186.3 tot/s, 105.4ms TTFT,   15.7ms ITL, … │
│ [0… throu… (… Req:    3.3 req/s,   52.61s Lat,   186.7 Conc,     213 Comp, … │
│               Tok:  324.5 gen/s, 1014.2 tot/s, 24183.8ms TTFT,  451.1ms ITL… │
│ [0… const… (… Req:    1.2 req/s,    1.22s Lat,     1.5 Conc,      72 Comp, … │
│               Tok:   76.7 gen/s,  239.8 tot/s, 118.8ms TTFT,   17.5ms ITL, … │
│ [0… const… (… Req:    1.5 req/s,    1.25s Lat,     1.9 Conc,      90 Comp, … │
│               Tok:   95.9 gen/s,  299.6 tot/s, 118.0ms TTFT,   17.9ms ITL, … │
│ [0… const… (… Req:    1.8 req/s,    1.34s Lat,     2.4 Conc,     107 Comp, … │
│               Tok:  114.7 gen/s,  358.6 tot/s, 119.6ms TTFT,   19.4ms ITL, … │
│ [0… const… (… Req:    2.1 req/s,    1.38s Lat,     2.9 Conc,     125 Comp, … │
│               Tok:  133.7 gen/s,  417.9 tot/s, 118.6ms T

In [9]:
!vllm bench serve --backend vllm --model Qwen/Qwen3-4B-AWQ \
  --endpoint /v1/completions --dataset-name random \
  --random-input-len 128 --random-output-len 64 --num-prompts 64 --save-result

Namespace(subparser='bench', bench_type='serve', dispatch_function=<function BenchmarkServingSubcommand.cmd at 0x7b5abd3b0e00>, trust_remote_code=False, seed=0, num_prompts=64, dataset_name='random', no_stream=False, dataset_path=None, no_oversample=False, skip_chat_template=False, enable_multimodal_chat=False, disable_shuffle=False, custom_output_len=256, custom_ensure_client_side_data=False, spec_bench_output_len=256, spec_bench_category=None, sonnet_input_len=550, sonnet_output_len=150, sonnet_prefix_len=200, sharegpt_output_len=None, timed_trace_chunk_hash_size=16, timed_trace_sec_multiplier=1, timed_trace_label_timestamp='timestamp', timed_trace_label_input_length='input_length', timed_trace_label_output_length='output_length', timed_trace_label_hash_ids='hash_ids', blazedit_min_distance=0.0, blazedit_max_distance=1.0, asr_max_audio_len_sec=inf, asr_min_audio_len_sec=0.0, random_input_len=128, random_output_len=64, random_range_ratio='0.0', random_prefix_len=0, random_batch_size=1